In [4]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn import Module, Linear, ReLU, Sequential, ModuleList
from torch_geometric.nn import TransformerConv, GatedGraphConv, BatchNorm
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import AddRandomWalkPE
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import pickle
import os
import zipfile
import torch_geometric
import h5py

# Initialize CUDA context explicitly
if torch.cuda.is_available():
    torch.cuda.init()
    print(f"CUDA initialized. Available GPUs: {torch.cuda.device_count()}")

# Print torch_geometric version
print(f"torch_geometric version: {torch_geometric.__version__}")

# Fix random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Set device for multi-GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
else:
    print("Using 1 GPU or CPU")

CUDA initialized. Available GPUs: 2
torch_geometric version: 2.6.1
Using 2 GPUs!


In [5]:
data_dir = 'PEMS-BAY'
h5_path = os.path.join(data_dir, 'PEMS-BAY.h5')
if not os.path.exists(h5_path):
    raise FileNotFoundError(f"PEMS-BAY dataset not found at {h5_path}")
print(f"Data directory {data_dir} ready")

# Verify adjacency matrix files
adj_matrices = [
    'adj_mx_knn_lane_pems.pkl',
    'adj_mx_knn_pems.pkl',
    'adj_mx_radius_lane_pems.pkl',
    'adj_mx_radius_pems.pkl'
]
for adj_file in adj_matrices:
    if not os.path.exists(os.path.join(data_dir, adj_file)):
        raise FileNotFoundError(f"Adjacency matrix {adj_file} not found in {data_dir}")

# Inspect HDF5 file structure
with h5py.File(h5_path, 'r') as f:
    print("Keys in HDF5 file:", list(f.keys()))
    print("\nContents of 'speed' group:")
    f['speed'].visititems(lambda name, obj: print(f"{name}: {type(obj)}" + (f", Shape: {obj.shape}, Dtype: {obj.dtype}" if isinstance(obj, h5py.Dataset) else "")))

Data directory PEMS-BAY ready
Keys in HDF5 file: ['speed']

Contents of 'speed' group:
axis0: <class 'h5py._hl.dataset.Dataset'>, Shape: (325,), Dtype: int64
axis1: <class 'h5py._hl.dataset.Dataset'>, Shape: (52116,), Dtype: int64
block0_items: <class 'h5py._hl.dataset.Dataset'>, Shape: (325,), Dtype: int64
block0_values: <class 'h5py._hl.dataset.Dataset'>, Shape: (52116, 325), Dtype: float64


In [6]:
def load_adjacency_matrix(adj_matrix_path, data_dir='PEMS-BAY'):
    try:
        with open(adj_matrix_path, 'rb') as f:
            adj_matrix = pickle.load(f)
        
        if isinstance(adj_matrix, tuple):
            for i in range(len(adj_matrix)):
                temp_matrix = np.array(adj_matrix[i])
                if temp_matrix.ndim == 2 and np.issubdtype(temp_matrix.dtype, np.number):
                    adj_matrix = temp_matrix
                    break
            else:
                raise ValueError(f"No 2D numeric array found in tuple: {adj_matrix}")
        elif isinstance(adj_matrix, dict):
            possible_keys = ['adj', 'adj_mx', 'matrix', 'adj_matrix', 'connectivity']
            for key in possible_keys:
                if key in adj_matrix:
                    adj_matrix = adj_matrix[key]
                    break
            else:
                raise ValueError(f"Adjacency matrix not found in dict. Keys found: {list(adj_matrix.keys())}")
        elif isinstance(adj_matrix, list):
            adj_matrix = np.array(adj_matrix)
        elif not isinstance(adj_matrix, np.ndarray):
            raise ValueError(f"Unexpected type for adjacency matrix: {type(adj_matrix)}")
        
        adj_matrix = np.array(adj_matrix, dtype=np.float32)
        
        if adj_matrix.ndim != 2:
            raise ValueError(f"Expected 2D adjacency matrix, got shape {adj_matrix.shape}")
        if adj_matrix.shape[0] != adj_matrix.shape[1]:
            raise ValueError(f"Expected square adjacency matrix, got shape {adj_matrix.shape}")
        if adj_matrix.shape[0] != 325:
            raise ValueError(f"Expected 325x325 adjacency matrix for PEMS-BAY, got shape {adj_matrix.shape}")
        
        if not np.issubdtype(adj_matrix.dtype, np.number):
            raise ValueError(f"Adjacency matrix contains non-numeric values, dtype: {adj_matrix.dtype}")
        
        non_zero_count = np.count_nonzero(adj_matrix)
        print(f"Number of non-zero entries in adjacency matrix: {non_zero_count}")
        if non_zero_count == 0:
            raise ValueError("Adjacency matrix has no non-zero entries, cannot create edge_index")
        
        print(f"Loaded adjacency matrix from {adj_matrix_path}, shape: {adj_matrix.shape}, dtype: {adj_matrix.dtype}")
        return adj_matrix
    
    except Exception as e:
        print(f"Error loading adjacency matrix from {adj_matrix_path}: {str(e)}")
        raise

In [7]:
def load_metr_la_data(data_dir='PEMS-BAY', num_days=99, custom_adj_path=None):
    if custom_adj_path is not None:
        print(f"Loading custom adjacency matrix from: {custom_adj_path}")
        adj_mat = load_adjacency_matrix(custom_adj_path, data_dir)
    else:
        raise ValueError("Custom adjacency matrix path must be provided for PEMS-BAY")
    
    print(f"Adjacency matrix shape: {adj_mat.shape}, dtype: {adj_mat.dtype}")

    # Load PEMS-BAY data from HDF5
    with h5py.File(os.path.join(data_dir, 'PEMS-BAY.h5'), 'r') as f:
        if 'speed/block0_values' not in f:
            raise KeyError("Dataset 'speed/block0_values' not found in HDF5 file")
        node_values = np.array(f['speed/block0_values'])
    print(f"Original node_values shape: {node_values.shape}")

    time_steps = num_days * 288
    if node_values.shape[0] != 52116:
        print(f"Warning: Expected 52116 timesteps, got {node_values.shape[0]}")
    if node_values.shape[0] < time_steps:
        raise ValueError(f"PEMS-BAY dataset has fewer than {time_steps} timesteps, got {node_values.shape[0]}")
    node_values = node_values[:time_steps]
    if node_values.ndim != 2:
        raise ValueError(f"Expected 2D node_values, got shape {node_values.shape}")
    if node_values.shape[1] != 325:
        raise ValueError(f"Expected 325 nodes for PEMS-BAY, got {node_values.shape[1]}")
    node_values = node_values.T  # Transpose to (325, time_steps)
    
    print(f"Processed node_values shape: {node_values.shape}")

    scaler = MinMaxScaler()
    node_values = scaler.fit_transform(node_values.T).T

    edge_index = np.where(adj_mat > 0)
    print(f"Raw edge_index from np.where: {len(edge_index)} arrays, shapes: {[arr.shape for arr in edge_index]}")
    if len(edge_index) != 2:
        raise ValueError(f"Expected edge_index to have 2 arrays (source, target), got {len(edge_index)}")
    edge_index = np.stack(edge_index, axis=0)
    print(f"Stacked edge_index shape: {edge_index.shape}")
    edge_index = torch.tensor(edge_index, dtype=torch.long)
    if edge_index.shape[0] != 2:
        raise ValueError(f"Expected edge_index to have shape [2, num_edges], got shape {edge_index.shape}")
    if edge_index.shape[1] == 0:
        raise ValueError("edge_index is empty; adjacency matrix may have no non-zero entries")
    edge_weight = torch.tensor(adj_mat[edge_index[0], edge_index[1]], dtype=torch.float)

    edge_index = edge_index.to(device)
    edge_weight = edge_weight.to(device)

    print(f"edge_index device: {edge_index.device}, shape: {edge_index.shape}")
    print(f"edge_weight device: {edge_weight.device}, shape: {edge_weight.shape}")

    return node_values, edge_index, edge_weight, scaler

In [8]:
def create_data_samples(node_values, in_timestep=12, out_timestep=12):
    features, labels = [], []
    for i in range(node_values.shape[1] - in_timestep - out_timestep + 1):
        features.append(node_values[:, i:i + in_timestep])
        labels.append(node_values[:, i + in_timestep:i + in_timestep + out_timestep])
    features = np.array(features, dtype=np.float32)
    labels = np.array(labels, dtype=np.float32)
    features = torch.tensor(features).permute(0, 2, 1)
    labels = torch.tensor(labels).permute(0, 2, 1)
    return features, labels

def create_100th_day_data(data_dir='PEMS-BAY', in_timestep=12, out_timestep=12):
    with h5py.File(os.path.join(data_dir, 'PEMS-BAY.h5'), 'r') as f:
        if 'speed/block0_values' not in f:
            raise KeyError("Dataset 'speed/block0_values' not found in HDF5 file")
        node_values = np.array(f['speed/block0_values'])
    print(f"100th day data: Original shape: {node_values.shape}")
    
    # Validate dataset size before slicing
    if node_values.shape[0] < 28512 + 288:
        print(f"Warning: Dataset has {node_values.shape[0]} timesteps, insufficient for 100th day. Using last 288 timesteps.")
        start_idx = max(0, node_values.shape[0] - 288)
        node_values = node_values[start_idx:start_idx + 288]
    else:
        print(f"Slicing timesteps 28512 to {28512 + 288} for 100th day")
        node_values = node_values[28512:28512 + 288]
    print(f"100th day sliced shape: {node_values.shape}")
    
    if node_values.shape[0] == 0:
        raise ValueError(f"100th day slice is empty, dataset size: {node_values.shape[0]}, indices: [28512:28512+288]")
    if node_values.ndim != 2:
        raise ValueError(f"Expected 2D node_values for 100th day, got shape {node_values.shape}")
    if node_values.shape[1] != 325:
        raise ValueError(f"Expected 325 nodes for PEMS-BAY 100th day, got {node_values.shape[1]}")
    node_values = node_values.T  # Transpose to (325, 288)
    print(f"100th day processed shape: {node_values.shape}")
    
    scaler_100 = MinMaxScaler()
    node_values = scaler_100.fit_transform(node_values.T).T
    features, labels = [], []
    for i in range(node_values.shape[1] - in_timestep - out_timestep + 1):
        features.append(node_values[:, i:i + in_timestep])
        labels.append(node_values[:, i + in_timestep:i + in_timestep + out_timestep])
    features = np.array(features, dtype=np.float32)
    labels = np.array(labels, dtype=np.float32)
    features = torch.tensor(features).permute(0, 2, 1)
    labels = torch.tensor(labels).permute(0, 2, 1)
    return features, labels, scaler_100

In [9]:
def compute_metrics(pred, target, scaler):
    batch_size, num_nodes, out_timestep = pred.shape
    pred_np = pred.cpu().numpy().transpose(0, 2, 1).reshape(-1, num_nodes)
    target_np = target.cpu().numpy().transpose(0, 2, 1).reshape(-1, num_nodes)
    pred_unscaled = scaler.inverse_transform(pred_np).reshape(batch_size, out_timestep, num_nodes).transpose(0, 2, 1)
    target_unscaled = scaler.inverse_transform(target_np).reshape(batch_size, out_timestep, num_nodes).transpose(0, 2, 1)
    pred_unscaled = torch.tensor(pred_unscaled, device=pred.device)
    target_unscaled = torch.tensor(target_unscaled, device=target.device)
    mae = torch.mean(torch.abs(pred_unscaled - target_unscaled)).item()
    rmse = torch.sqrt(torch.mean((pred_unscaled - target_unscaled) ** 2)).item()
    mask = target_unscaled > 1.0
    mape = torch.where(mask, torch.abs((pred_unscaled - target_unscaled) / (target_unscaled + 1e-2)), 0.0)
    mape = torch.mean(mape[mask]) * 100 if mask.sum() > 0 else 0.0
    return mae, rmse, mape

def visualize_predictions(pred, target, scaler, output_dir='plots', num_sensors=5, filename='prediction_100th_day.png'):
    os.makedirs(output_dir, exist_ok=True)
    pred = pred[0].cpu().numpy().T
    target = target[0].cpu().numpy().T
    pred = scaler.inverse_transform(pred).T
    target = scaler.inverse_transform(target).T
    plt.figure(figsize=(12, 10))
    for i in range(min(num_sensors, pred.shape[0])):
        plt.subplot(num_sensors, 1, i + 1)
        plt.plot(pred[i], label='Predicted', marker='o')
        plt.plot(target[i], label='Actual', marker='x')
        plt.title(f'Sensor {i + 1} Traffic Speed Prediction (100th Day)')
        plt.xlabel('Time Step (5-min intervals)')
        plt.ylabel('Speed (mph)')
        plt.legend()
        plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, filename))
    plt.close()

def visualize_full_100th_day(day_100_loader, model, scaler, output_dir='plots', num_sensors=5, filename='full_100th_day_prediction.png'):
    model.eval()
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for data in day_100_loader:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.edge_attr, data.pe if hasattr(data, 'pe') else None, data.batch)
            batch_size = data.batch.max().item() + 1
            num_nodes = out.shape[1]
            target = data.y.view(batch_size, num_nodes, -1)
            all_preds.append(out.cpu().numpy())
            all_targets.append(target.cpu().numpy())

    all_preds = np.concatenate(all_preds, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)

    num_nodes = all_preds.shape[1]
    full_preds = np.zeros((num_nodes, 288))
    full_targets = np.zeros((num_nodes, 288))
    counts = np.zeros((num_nodes, 288))

    for i in range(all_preds.shape[0]):
        start_idx = i + 12
        end_idx = start_idx + 12
        full_preds[:, start_idx:end_idx] += all_preds[i]
        full_targets[:, start_idx:end_idx] += all_targets[i]
        counts[:, start_idx:end_idx] += 1

    mask = counts > 0
    full_preds[mask] = full_preds[mask] / counts[mask]
    full_targets[mask] = full_targets[mask] / counts[mask]

    full_preds = scaler.inverse_transform(full_preds.T).T
    full_targets = scaler.inverse_transform(full_targets.T).T

    covered_time_steps = slice(12, 277)
    time_steps = np.arange(12, 277) * 5 / 60
    os.makedirs(output_dir, exist_ok=True)
    plt.figure(figsize=(15, 10))
    for i in range(min(num_sensors, num_nodes)):
        plt.subplot(num_sensors, 1, i + 1)
        plt.plot(time_steps, full_preds[i, covered_time_steps], label='Predicted', marker='o', markersize=3)
        plt.plot(time_steps, full_targets[i, covered_time_steps], label='Actual', marker='x', markersize=3)
        plt.title(f'Sensor {i + 1} Traffic Speed Prediction (100th Day, Hours 1–23)')
        plt.xlabel('Time (hours)')
        plt.ylabel('Speed (mph)')
        plt.legend()
        plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, filename))
    plt.close()

In [10]:
import torch
import torch.nn as nn
from torch_geometric.nn import TransformerConv, GatedGraphConv, LayerNorm

class GPSLayer(nn.Module):
    def __init__(self, hidden_channels, heads=8, dropout=0.1):  # Reduced dropout from 0.3 to 0.1
        super(GPSLayer, self).__init__()
        self.gated_gcn = GatedGraphConv(hidden_channels, num_layers=1)
        self.transformer = TransformerConv(
            hidden_channels, hidden_channels, heads=heads, concat=False, dropout=dropout
        )
        self.norm1 = LayerNorm(hidden_channels)  # Split normalization into two steps
        self.norm2 = LayerNorm(hidden_channels)
        self.ffn = nn.Sequential(  # Added feed-forward network
            nn.Linear(hidden_channels, hidden_channels * 2),
            nn.ReLU(),
            nn.Linear(hidden_channels * 2, hidden_channels),
            nn.Dropout(dropout)
        )
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, edge_index, edge_weight, batch):
        # Gated GCN path
        x_gcn = self.gated_gcn(x, edge_index, edge_weight=edge_weight)
        x_gcn = self.norm1(x_gcn)  # Normalize after GCN
        # Transformer path
        x_trans = self.transformer(x_gcn, edge_index)
        x = x_gcn + x_trans + x  # Residual connection
        x = self.norm2(x)  # Normalize after residual
        x = torch.relu(x)
        x = self.dropout(x)
        # Feed-forward network
        x_ffn = self.ffn(x)
        x = x + x_ffn  # Residual connection for FFN
        return x

class GPSGraphTransformer(nn.Module):
    def __init__(self, in_channels, hidden_channels=128, out_channels=12, num_layers=3, pe_dim=20, heads=4, use_pe=True):  # Changed num_layers to 3, heads to 4
        super().__init__()
        self.use_pe = use_pe
        self.node_emb = nn.Linear(in_channels, hidden_channels - pe_dim if use_pe else hidden_channels)
        if use_pe:
            self.pe_lin = nn.Linear(pe_dim, pe_dim)
        self.layers = nn.ModuleList([GPSLayer(hidden_channels, heads=heads, dropout=0.1) for _ in range(num_layers)])  # Updated dropout
        self.norm = LayerNorm(hidden_channels)
        self.mlp = nn.Sequential(  # Enhanced MLP
            nn.Linear(hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_channels // 2, out_channels)
        )
    
    def forward(self, x, edge_index, edge_weight, pe=None, batch=None):
        x = self.node_emb(x)
        if self.use_pe and pe is not None:
            pe = self.pe_lin(pe)
            x = torch.cat([x, pe], dim=-1)
        for layer in self.layers:
            x = layer(x, edge_index, edge_weight, batch)
        x = self.norm(x)
        x = self.mlp(x)
        batch_size = batch.max().item() + 1 if batch is not None else 1
        num_nodes = x.size(0) // batch_size
        return x.view(batch_size, num_nodes, -1)


In [11]:
# Main workflow for single adjacency matrix
adj_matrices = [
    'adj_mx_knn_lane_pems.pkl',
    'adj_mx_knn_pems.pkl',
    'adj_mx_radius_lane_pems.pkl',
    'adj_mx_radius_pems.pkl'
]
results = {}

# Display available adjacency matrices
print("Available adjacency matrices:")
for i, adj_file in enumerate(adj_matrices, 1):
    print(f"{i}. {adj_file}")

# Prompt user for input
while True:
    try:
        choice = input("Enter the number (1-4) of the adjacency matrix to use: ")
        choice = int(choice)
        if 1 <= choice <= 4:
            adj_file = adj_matrices[choice - 1]
            break
        else:
            print("Invalid choice. Please enter a number between 1 and 4.")
    except ValueError:
        print("Invalid input. Please enter a number between 1 and 4.")

print(f"\nProcessing with adjacency matrix: {adj_file}")

try:
    # Load data with the selected adjacency matrix
    node_values, edge_index, edge_weight, scaler = load_metr_la_data(custom_adj_path=f'PEMS-BAY/{adj_file}')
    
    # Create data samples
    features, labels = create_data_samples(node_values)
    
    # Split data into train, validation, and test sets
    total_samples = len(features)
    train_end = int(0.6 * total_samples)
    val_end = train_end + int(0.2 * total_samples)
    
    train_features = features[:train_end]
    train_labels = labels[:train_end]
    val_features = features[train_end:val_end]
    val_labels = labels[train_end:val_end]
    test_features = features[val_end:]
    test_labels = labels[val_end:]
    
    print(f"Train samples: {len(train_features)}, Val samples: {len(val_features)}, Test samples: {len(test_features)}")
    
    # Apply random walk positional encoding
    transform = AddRandomWalkPE(walk_length=20, attr_name='pe')
    
    # Create training data
    train_data = []
    for i in range(len(train_features)):
        data = Data(x=train_features[i].T, y=train_labels[i].T, edge_index=edge_index, edge_attr=edge_weight)
        data = transform(data)
        train_data.append(data)
    
    # Create validation data
    val_data = []
    for i in range(len(val_features)):
        data = Data(x=val_features[i].T, y=val_labels[i].T, edge_index=edge_index, edge_attr=edge_weight)
        data = transform(data)
        val_data.append(data)
    
    # Create test data
    test_data = []
    for i in range(len(test_features)):
        data = Data(x=test_features[i].T, y=test_labels[i].T, edge_index=edge_index, edge_attr=edge_weight)
        data = transform(data)
        test_data.append(data)
    
    # Create 100th day data
    day_100_features, day_100_labels, day_100_scaler = create_100th_day_data()
    day_100_data = []
    for i in range(len(day_100_features)):
        data = Data(x=day_100_features[i].T, y=day_100_labels[i].T, edge_index=edge_index, edge_attr=edge_weight)
        data = transform(data)
        if 'pe' not in data:
            print(f"Warning: Positional encoding 'pe' not added to Day 100 Data {i}; proceeding without PE")
        day_100_data.append(data)
    
    # Create data loaders
    train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_data, batch_size=32, shuffle=False)
    day_100_loader = DataLoader(day_100_data, batch_size=32, shuffle=False)
    
    # Initialize model
    model = GPSGraphTransformer(in_channels=12, hidden_channels=64, out_channels=12, num_layers=4, pe_dim=20, heads=4, use_pe=True)
    model = model.to(device)
    
    # Optimizer and loss function
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
    criterion = nn.MSELoss()
    
    # Training loop
    for epoch in range(20):
        model.train()
        total_train_loss = 0
        for data in train_loader:
            data = data.to(device)
            if data.edge_index.shape[0] != 2:
                raise ValueError(f"Invalid edge_index shape before model call: {data.edge_index.shape}")
            optimizer.zero_grad()
            out = model(data.x, data.edge_index, data.edge_attr, data.pe if hasattr(data, 'pe') else None, data.batch)
            batch_size = data.batch.max().item() + 1
            num_nodes = out.shape[1]
            target = data.y.view(batch_size, num_nodes, -1)
            loss = criterion(out, target)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
        train_loss = total_train_loss / len(train_loader)
        
        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for data in val_loader:
                data = data.to(device)
                if data.edge_index.shape[0] != 2:
                    raise ValueError(f"Invalid edge_index shape in val_loader: {data.edge_index.shape}")
                out = model(data.x, data.edge_index, data.edge_attr, data.pe if hasattr(data, 'pe') else None, data.batch)
                batch_size = data.batch.max().item() + 1
                num_nodes = out.shape[1]
                target = data.y.view(batch_size, num_nodes, -1)
                loss = criterion(out, target)
                total_val_loss += loss.item()
        val_loss = total_val_loss / len(val_loader)
        print(f"Epoch {epoch + 1}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
    
    # Evaluate on test set
    model.eval()
    total_mae, total_rmse, total_mape = 0, 0, 0
    num_batches = 0
    with torch.no_grad():
        for data in test_loader:
            data = data.to(device)
            if data.edge_index.shape[0] != 2:
                raise ValueError(f"Invalid edge_index shape in test_loader: {data.edge_index.shape}")
            out = model(data.x, data.edge_index, data.edge_attr, data.pe if hasattr(data, 'pe') else None, data.batch)
            batch_size = data.batch.max().item() + 1
            num_nodes = out.shape[1]
            target = data.y.view(batch_size, num_nodes, -1)
            mae, rmse, mape = compute_metrics(out, target, scaler)
            total_mae += mae
            total_rmse += rmse
            total_mape += mape
            num_batches += 1
            if num_batches == 1:
                visualize_predictions(out, target, scaler, output_dir='plots_test', filename=f'prediction_test_{adj_file}.png')
        test_mae = total_mae / num_batches
        test_rmse = total_rmse / num_batches
        test_mape = total_mape / num_batches
        print(f"Test MAE: {test_mae:.4f} mph, RMSE: {test_rmse:.4f} mph, MAPE: {test_mape:.4f}%")
    
    # Evaluate on 100th day
    total_mae, total_rmse, total_mape = 0, 0, 0
    num_batches = 0
    with torch.no_grad():
        for data in day_100_loader:
            data = data.to(device)
            if data.edge_index.shape[0] != 2:
                raise ValueError(f"Invalid edge_index shape in day_100_loader: {data.edge_index.shape}")
            out = model(data.x, data.edge_index, data.edge_attr, data.pe if hasattr(data, 'pe') else None, data.batch)
            batch_size = data.batch.max().item() + 1
            num_nodes = out.shape[1]
            target = data.y.view(batch_size, num_nodes, -1)
            mae, rmse, mape = compute_metrics(out, target, day_100_scaler)
            total_mae += mae
            total_rmse += rmse
            total_mape += mape
            num_batches += 1
            if num_batches == 1:
                visualize_predictions(out, target, day_100_scaler, filename=f'prediction_100th_day_{adj_file}.png')
        day_100_mae = total_mae / num_batches
        day_100_rmse = total_rmse / num_batches
        day_100_mape = total_mape / num_batches
        print(f"100th Day MAE: {day_100_mae:.4f} mph, RMSE: {day_100_rmse:.4f} mph, MAPE: {day_100_mape:.4f}%")
    
    # Visualize full 100th day
    visualize_full_100th_day(day_100_loader, model, day_100_scaler, filename=f'full_100th_day_prediction_{adj_file}.png')
    
    # Store results
    results[adj_file] = {
        'test_mae': test_mae, 'test_rmse': test_rmse, 'test_mape': test_mape,
        'day_100_mae': day_100_mae, 'day_100_rmse': day_100_rmse, 'day_100_mape': day_100_mape
    }

except Exception as e:
    print(f"Error processing {adj_file}: {str(e)}")
    raise


Available adjacency matrices:
1. adj_mx_knn_lane_pems.pkl
2. adj_mx_knn_pems.pkl
3. adj_mx_radius_lane_pems.pkl
4. adj_mx_radius_pems.pkl



Processing with adjacency matrix: adj_mx_radius_lane_pems.pkl
Loading custom adjacency matrix from: PEMS-BAY/adj_mx_radius_lane_pems.pkl
Number of non-zero entries in adjacency matrix: 614
Loaded adjacency matrix from PEMS-BAY/adj_mx_radius_lane_pems.pkl, shape: (325, 325), dtype: float32
Adjacency matrix shape: (325, 325), dtype: float32
Original node_values shape: (52116, 325)
Processed node_values shape: (325, 28512)
Raw edge_index from np.where: 2 arrays, shapes: [(614,), (614,)]
Stacked edge_index shape: (2, 614)
edge_index device: cuda:0, shape: torch.Size([2, 614])
edge_weight device: cuda:0, shape: torch.Size([614])
Train samples: 17093, Val samples: 5697, Test samples: 5699
100th day data: Original shape: (52116, 325)
Slicing timesteps 28512 to 28800 for 100th day
100th day sliced shape: (288, 325)
100th day processed shape: (325, 288)
Epoch 1, Train Loss: 0.0306, Val Loss: 0.0075


KeyboardInterrupt: 